In [35]:
import pandas as pd
import json
import plotly.express as px
import numpy as np
import seaborn as sns
from tqdm import tqdm

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn import tree

from sklearn.linear_model import LinearRegression

from joblib import dump, load

import xgboost as xgb

The purpose of this notebook is to create the various joblib files with the selected parameters.

In [2]:
with open('../user_course_info_cleaned.json', 'r') as file:
    user_course_info = json.load(file)

df_user_course_info = pd.DataFrame(user_course_info)

In [3]:
with open('../teacher_stats_cleaned.json', 'r') as file:
    teacher_stats = json.load(file)

df_teacher_stats = pd.DataFrame(teacher_stats)

In [4]:
with open('../pro_course_cleaned.json', 'r') as file:
    pro_course = json.load(file)

df_pro_course = pd.DataFrame(pro_course)

In [5]:
with open('../price_list_cleaned.json', 'r') as file:
    price_list = json.load(file)

df_price_list = pd.DataFrame(price_list)

In [6]:
with open('../also_speaks_cleaned.json', 'r') as file:
    also_speaks = json.load(file)

df_also_speaks = pd.DataFrame(also_speaks)

In [7]:
with open('../also_speaks_reference.json', 'r') as file:
    also_speaks_reference = json.load(file)

df_also_speaks_reference = pd.DataFrame(also_speaks_reference)

Price Prediction Model Functions

In [8]:
def is_tutor(tutor):
    if tutor == True:
        df_tutors = df_user_course_info[(df_user_course_info['is_tutor'] == 1) & (df_user_course_info['is_pro']  == 0)]
        return df_tutors[['user_id', 'trial_session_count', 'trial_price']]
    else:
        df_pros_1 = df_user_course_info[(df_user_course_info['is_tutor'] == 1) & (df_user_course_info['is_pro']  == 1)]
        df_pros_2 = df_user_course_info[(df_user_course_info['is_tutor'] == 0) & (df_user_course_info['is_pro']  == 1)]
        df_pros_combined = pd.concat([df_pros_1, df_pros_2], ignore_index=True)
        df_pros_combined = df_pros_combined[['user_id', 'trial_session_count', 'trial_price']]
        return df_pros_combined
        
        
is_tutor(True)

,user_id,trial_session_count,trial_price
0,55502,97,1000
3,132815,0,750
6,249152,70,800
11,355886,142,500
12,368484,91,1000
...,...,...,...
10465,32167939,3,1000
10466,32168520,0,500
10467,32169200,0,1400
10468,32169731,0,600


In [9]:
def teacher_id_language(language):
    df_teachers_id = df_pro_course[df_pro_course['language'] == language].groupby(['teacher_id', 'language']).count()
    df_teachers_id = df_teachers_id.reset_index()
    df_teachers_id = df_teachers_id[['teacher_id']]
    return df_teachers_id

teacher_id_language('chinese')

,teacher_id
0,55502
1,113638
2,114708
3,259649
4,436016
...,...
993,31845535
994,31847889
995,31913484
996,31915782


In [10]:
def pro_course_language(language):
    df_pro_course_target = df_pro_course[df_pro_course['language'] == language]
    return df_pro_course_target

pro_course_language('german')

,id,teacher_id,language,title,session_price,student_count,session_count,has_package
11,9022,516278,german,** GERMAN PROFICIENCY ★ 50 X 90 min ► All leve...,6090,84,723,1
12,9024,516278,german,★ NEXT LEVEL GERMAN ★ 40 X 90 min ► All levels...,6090,70,278,1
20,10161,516278,german,★ GERMAN IMMERSION ★ 30 X 90 min ► All levels ...,6090,46,371,1
87,20714,516278,german,Konversation,2490,211,1368,1
103,21847,222305,german,German,1499,582,3571,1
...,...,...,...,...,...,...,...,...
38971,290636,31827412,german,Relaxed German Conversation Practice for All L...,1400,0,0,1
38980,290655,31079682,german,Alemán a través de la Literatura Germana: Goet...,5000,0,0,1
38981,290656,31079682,german,"Cultura, Literatura y Actualidad Alemana – Con...",3000,0,0,1
38994,290678,18952636,german,Das Kolloquium,5000,0,0,1


In [11]:
def language_is_tutor(language, tutor):        
    df_tutor_pro = is_tutor(tutor)
    df_language = pro_course_language(language)
    df_merged = pd.merge(df_tutor_pro, df_language, left_on='user_id', right_on='teacher_id', how='inner')
    # df_merged = df_merged[['user_id', 'trial_session_count', 'trial_price', 'student_count', 'session_count', 'has_package', 'session_price']]
    df_merged = df_merged[['teacher_id', 'title', 'trial_session_count', 'trial_price', 'student_count', 'session_count', 'has_package', 'session_price']]
    # df_merged = df_merged[['trial_price', 'session_price']]
    return df_merged

language_is_tutor('french', True)

,teacher_id,title,trial_session_count,trial_price,student_count,session_count,has_package,session_price
0,614209,Français,0,1000,0,0,1,3000
1,740680,Informal Tutoring,296,1500,435,5178,1,2000
2,740680,Français pour 2,296,1500,16,238,1,2500
3,867550,French conversational practice 10 lessons package,182,800,421,2738,1,1600
4,867550,Preparing DELF/DALF exam from level A2 to C2,182,800,31,143,1,1800
...,...,...,...,...,...,...,...,...
1926,32049068,Conversation décontractée et pratique,0,500,4,7,1,1000
1927,32117853,French for Beginners – Greetings and Self-Intr...,0,500,0,0,1,700
1928,32164431,Professional French Coaching | Personalized Le...,0,2500,0,0,1,5000
1929,32167939,French Beginner A1 to C2 proficient,3,1000,0,0,0,2000


The courses that have less than 10 sessions or 10 trial sessions should be removed\
Since their progress has not yet stabilized, this should not be included in the training and test data.

In [280]:
print(rmse, r2)

381.6736362859636 0.8081045435596184


In [268]:
#Do not use less than 10 students and 10 sessions
forest_depth = 20
minimum_stud = 10
minimum_session = minimum_stud * 4
tutor = True
language = 'korean'

df_model = language_is_tutor(language, tutor)
df_model = df_model[(df_model['session_count'] > minimum_session) & (df_model['trial_session_count'] > minimum_stud)]
df_model

,teacher_id,title,trial_session_count,trial_price,student_count,session_count,has_package,session_price
5,815733,Korean Conversation Practice – Speak Mostly in...,345,1500,74,538,1,3000
6,815733,"General Korean Class – From A to Z, For Everyone",345,1500,322,2045,1,3200
8,901015,Formal Korean Lessons,157,1400,30,225,1,2600
11,1509798,Informal Tutoring,125,1000,290,8076,1,1500
12,1745587,Korean absolute 0 to conversational,361,7000,51,307,1,8000
...,...,...,...,...,...,...,...,...
569,29325526,Free Talking Class : Sound Like a Native! Casu...,13,800,20,59,1,1500
615,30615695,✨[All-in-One] Master Real-Life Korean Conversa...,14,1000,18,222,1,2200
627,30685471,🥇Structured Korean for Real-Life Communication...,22,900,15,107,1,2000
641,30742203,"한국어 회화 및 문화 | 일상생활, 케이팝, 드라마 주제 | 편안하지만 체계적인 수업",20,500,52,264,1,1000


In [269]:
df_merge = pd.merge(df_user_course_info, df_pro_course, left_on='user_id', right_on='teacher_id', how='left')
df_merge_user = df_merge.groupby('user_id')[['student_count', 'session_count']].sum()
df_merge_session_price = df_merge.groupby('user_id')[['session_price']].mean()

df_merge_user = df_merge_user.reset_index()

df_merge_user_zero = df_merge_user[(df_merge_user['student_count'] > 10) & (df_merge_user['session_count'] > 10)]
df_merge_user_zero['session to student'] = df_merge_user_zero['session_count'] / df_merge_user_zero['student_count']
df_merge_user_zero = pd.merge(df_merge_user_zero, df_merge_session_price, left_on='user_id', right_on='user_id', how='left')
df_merge_user_zero = df_merge_user_zero[['user_id', 'session to student']]

df_join = pd.merge(df_model, df_merge_user_zero, left_on='teacher_id', right_on='user_id', how='left')
df_model = df_join[['trial_session_count', 'trial_price', 'has_package', 'session to student', 'session_price']]
df_model

,trial_session_count,trial_price,has_package,session to student,session_price
0,345,1500,1,6.400922,3000
1,345,1500,1,6.400922,3200
2,157,1400,1,6.742222,2600
3,125,1000,1,27.848276,1500
4,361,7000,1,9.047146,8000
...,...,...,...,...,...
147,13,800,1,3.404255,1500
148,14,1000,1,12.333333,2200
149,22,900,1,5.923077,2000
150,20,500,1,5.076923,1000


In [270]:
features = list(df_model.columns)
# features.remove('title')
features.remove('session_price')
features

['trial_session_count', 'trial_price', 'has_package', 'session to student']

In [271]:
X = df_model[features]
y = df_model['session_price']

In [272]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [273]:
regr = RandomForestRegressor(n_estimators=100, max_depth=forest_depth, random_state=42)
regr.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [274]:
y_pred = regr.predict(X_test)

In [275]:
rmse = root_mean_squared_error(y_pred, y_test)

In [276]:
r2 = r2_score(y_test,y_pred)
r2

0.8081045435596184

In [277]:
regr.predict([[10, 500, 0, 6]])

c:\Users\pecke\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


array([1036.89])

In [278]:
if tutor == True:
    file_extension = 'tutor'
else:
    file_extension = 'pro'

In [279]:
dump(regr, f'{language}_{file_extension}.joblib')

['korean_tutor.joblib']